# XGBoost and logistic regression comparison

This experiment fits **two independent binary classifiers**: a subsequent shot,
and a same-team goal before possession ends. Both use the existing states
**before the first explicit shot**, exclusions and possession weights.

The comparison evaluates whether boosted trees improve probability estimates
from the **same five current-state inputs**. Holding the input features fixed
supports a comparison of linear and nonlinear models, including interactions
between action, location and timing. Neither model uses action history.

Candidates fit on **2023–2024**, **2025** selects hyperparameters and tree count,
and fresh models refit on **2023–2025** before evaluating **2026**. That test season
has already been observed for logistic regression; it is a shared benchmark.

Run `python -m afl_opposition.xgboost_baseline` after both tuned logistic experiments
to generate these artifacts. This notebook **only reads saved results**. Saved
outputs are included; run cells in order with **Shift+Enter** using the project
kernel. See [the experiment specification](../docs/xgboost-baseline.md).

## 1. Experiment verification

The loader resolves paths relative to the repository and verifies model,
search-table and report checksums against the completion record before
reading the results. A missing completion record means training has not completed
successfully; rerunning this notebook does not start it.

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/sources.json").is_file() and (p / "pyproject.toml").is_file()), None)
if project_root is None:
    raise RuntimeError("Launch from the repository root or notebooks folder.")
experiment = project_root / "artifacts/xgboost-2026"
if not (experiment / "summary.json").is_file():
    raise RuntimeError("Run both tuned logistic experiments, then python -m afl_opposition.xgboost_baseline.")
summary = json.loads((experiment / "summary.json").read_text())
for relative, expected in summary["artifact_sha256"].items():
    with (experiment / relative).open("rb") as stream:
        assert hashlib.file_digest(stream, "sha256").hexdigest() == expected, relative
targets = summary["targets"]
refits = {target: json.loads((experiment / target / "refit.json").read_text()) for target in targets}
print("Experiment:", experiment.relative_to(project_root).as_posix())
print("Verified artifacts:", len(summary["artifact_sha256"]))
print("XGBoost version:", summary["xgboost_version"])

Experiment: artifacts/xgboost-2026
Verified artifacts: 62
XGBoost version: 3.4.1


## 2. Model architecture and inputs

XGBoost adds decision trees sequentially. Each new tree uses first and second
derivatives of binary cross-entropy to improve the current predictions. The
ensemble's score passes through a sigmoid:

$$p_i = \sigma\!\left(b + \eta\sum_{t=1}^{T} f_t(x_i)\right).$$

A tree can learn interactions such as different position effects for kicks and
handballs. The baseline's one-hot action encoding and training-fitted numeric
scaler are reused. Trees do not require scaling; retaining it keeps the feature
pipeline consistent. No label, shot flag, eventual outcome or future coordinate
is an input.

There are no past action identities in the model. `gap_seconds` and
`elapsed_seconds` summarise timing already observed at the current action.

In [2]:
display(pd.DataFrame([
    ("action_type", "Categorical", "One-hot encoded current action"),
    ("x_norm", "Continuous", "Current position along the actor's attacking direction"),
    ("y_norm", "Continuous", "Current lateral position"),
    ("gap_seconds", "Continuous", "Time since the previous retained action"),
    ("elapsed_seconds", "Continuous", "Time since the inferred possession began"),
], columns=["feature", "type", "meaning"]))
for target in targets:
    print(target, "encoded input count:", len(refits[target]["encoded_features"]))
display(pd.DataFrame([
    ("Fit candidates and preprocessing", "2023–2024"),
    ("Choose hyperparameters and number of trees", "2025 weighted BCE"),
    ("Refit fresh models and preprocessing", "2023–2025"),
    ("Evaluate frozen predictions", "2026"),
], columns=["step", "data"]))

,feature,type,meaning
0,action_type,Categorical,One-hot encoded current action
1,x_norm,Continuous,Current position along the actor's attacking d...
2,y_norm,Continuous,Current lateral position
3,gap_seconds,Continuous,Time since the previous retained action
4,elapsed_seconds,Continuous,Time since the inferred possession began


shot encoded input count: 16
goal encoded input count: 16


,step,data
0,Fit candidates and preprocessing,2023–2024
1,Choose hyperparameters and number of trees,2025 weighted BCE
2,Refit fresh models and preprocessing,2023–2025
3,Evaluate frozen predictions,2026


## 3. Validation search

The fixed grid contains 18 candidates per target: depth 2/4/6, learning rate
0.05/0.1 and native L2 parameter 0/1/10. Every candidate starts fresh, can fit up
to 1,200 trees, and stops after 60 rounds without improvement in **weighted
validation BCE**. Validation weights also give each possession total weight one.

The saved booster retains its best iteration, rather than the last attempted
iteration. The winner has minimum weighted BCE; exact ties prefer fewer trees,
shallower trees, stronger L2, then earlier trial order. AP and Brier are diagnostics.
`best_at_round_cap` flags a search bounded by the declared tree limit.

In [3]:
searches = {target: pd.read_parquet(experiment / target / "search.parquet") for target in targets}
search = pd.concat([table.assign(target=target) for target, table in searches.items()], ignore_index=True)
for target in targets:
    table = searches[target]
    assert table.selected.sum() == 1
    assert table.loc[table.selected, "validation_log_loss"].iloc[0] == table.validation_log_loss.min()
display(search[["target", "trial", "max_depth", "eta", "reg_lambda", "rounds", "attempted_rounds",
                "validation_log_loss", "validation_average_precision", "validation_brier_score", "selected", "best_at_round_cap"]]
        .sort_values(["target", "validation_log_loss"]).reset_index(drop=True).style.format({
            "validation_log_loss": "{:.6f}", "validation_average_precision": "{:.6f}", "validation_brier_score": "{:.6f}"}))

,target,trial,max_depth,eta,reg_lambda,rounds,attempted_rounds,validation_log_loss,validation_average_precision,validation_brier_score,selected,best_at_round_cap
0,goal,17,6,0.100000,10.000000,120,180,0.277795,0.333631,0.078978,True,False
1,goal,13,6,0.050000,1.000000,198,258,0.277828,0.333497,0.078982,False,False
2,goal,14,6,0.050000,10.000000,264,324,0.277838,0.333562,0.078991,False,False
3,goal,9,4,0.100000,0.000000,218,278,0.277838,0.333392,0.078997,False,False
4,goal,16,6,0.100000,1.000000,79,139,0.277858,0.333257,0.078993,False,False
5,goal,12,6,0.050000,0.000000,159,219,0.277858,0.333281,0.078991,False,False
6,goal,7,4,0.050000,1.000000,426,486,0.277861,0.332994,0.079003,False,False
7,goal,11,4,0.100000,10.000000,248,308,0.277865,0.332931,0.079014,False,False
8,goal,6,4,0.050000,0.000000,385,445,0.277870,0.332906,0.079014,False,False
9,goal,8,4,0.050000,10.000000,601,661,0.277874,0.332961,0.079013,False,False


## 4. Selected-model learning curves

These are training and validation losses from the **candidate fitted on 2023–2024**.
The table shows the first round, every hundredth round, the selected round and the
last attempted round. It is a sample of the complete saved learning curve.

Training loss can keep falling while validation loss stops improving. The
selected tree count is retained even though early stopping needs later rounds
to establish that progress has stopped. Test data do not determine this count.

In [4]:
for target in targets:
    chosen = refits[target]["selected"]
    curve = pd.read_parquet(experiment / target / "learning_curves.parquet")
    curve = curve.loc[curve.trial.eq(chosen["trial"])].pivot(index="round", columns="split", values="log_loss")
    keep = (curve.index % 100 == 0) | curve.index.isin([1, chosen["rounds"], chosen["attempted_rounds"]])
    shown = curve.loc[keep].copy()
    shown["selected_round"] = shown.index == chosen["rounds"]
    print(target, "— selected trees:", chosen["rounds"])
    display(shown.style.format({"train": "{:.6f}", "validation": "{:.6f}"}))

shot — selected trees: 147


split,train,validation,selected_round
round,,,
1,0.475023,0.484417,False
100,0.373825,0.387631,False
147,0.372872,0.387493,True
200,0.371828,0.387588,False
207,0.371742,0.387609,False


goal — selected trees: 120


split,train,validation,selected_round
round,,,
1,0.310162,0.316439,False
100,0.268749,0.277830,False
120,0.268448,0.277795,True
180,0.267708,0.277843,False


## 5. Refitting and regularisation

The selected depth, learning rate and tree count define a fresh fit on 2023–2025.
There is **no early stopping during refitting**, because validation is now part of
the fitting data. The encoder, scaler and initial class prevalence are refitted.

XGBoost's native objective sums weighted BCE and adds
$\lambda_{\rm native}\sum_{t,j}w_{tj}^2/2$ for leaf values $w_{tj}$.
Relative to mean weighted BCE, its coefficient is
$\lambda_{\rm native}/(2S)$ where $S=\sum_i s_i$.
The refit increases the native L2 parameter by $S_{\rm refit}/S_{\rm train}$,
preserving this coefficient. The minimum child Hessian is scaled by the same
ratio. This parameter is distinct from the logistic model's coefficient penalty.

All model files are saved and hashed before test loading. Both refits must finish
before either model is evaluated on 2026.

In [5]:
records = []
for target, frozen in refits.items():
    selected, final = frozen["selection_parameters"], frozen["refit_parameters"]
    assert abs(selected["reg_lambda"] / (2 * frozen["training_weight_sum"])
               - final["reg_lambda"] / (2 * frozen["refit_weight_sum"])) < 1e-12
    assert frozen["rounds"] == frozen["selected"]["rounds"]
    records.append(dict(target=target, depth=final["max_depth"], learning_rate=final["eta"], trees=frozen["rounds"],
        training_weight=frozen["training_weight_sum"], refit_weight=frozen["refit_weight_sum"],
        selected_native_L2=selected["reg_lambda"], refit_native_L2=final["reg_lambda"],
        mean_loss_L2=frozen["mean_loss_l2"], constant_probability=frozen["constant_probability"]))
display(pd.DataFrame(records))
assert summary["both_models_frozen_before_test_loading"] and summary["frozen_models_unchanged"]
print("Both refits frozen before test loading; model hashes unchanged after evaluation.")

,target,depth,learning_rate,trees,training_weight,refit_weight,selected_native_L2,refit_native_L2,mean_loss_L2,constant_probability
0,shot,6,0.1,147,103078.0,153822.0,1.0,1.492287,0.000005,0.199692
1,goal,6,0.1,120,103078.0,153822.0,10.0,14.922874,0.000049,0.098555


Both refits frozen before test loading; model hashes unchanged after evaluation.


## 6. Test coverage and weighting

Each row is a pre-shot event state. A possession with `n` eligible states gives
each state weight `1/n`, so every possession contributes total weight one.
Unweighted positive prevalence can therefore differ from the weighted prevalence.
These are correlated states within matches, not independent matches.

The experiment checks that each target uses exactly the same event IDs, labels,
weights and constant probabilities as its logistic comparison. Shot and goal
labels differ: a behind is shot-positive and goal-negative.

In [6]:
predictions = {target: pd.read_parquet(experiment / target / "test/predictions.parquet") for target in targets}
coverage = []
for target, table in predictions.items():
    weights = table.groupby("possession_id").sample_weight.sum()
    assert weights.sub(1).abs().max() < 1e-12
    coverage.append(dict(target=target, matches=table.match_id.nunique(), prefixes=len(table),
        possessions=len(weights), positive_possessions=table.groupby("possession_id").target.first().sum(),
        unweighted_positive_rate=table.target.mean(),
        weighted_positive_rate=(table.target * table.sample_weight).sum() / table.sample_weight.sum()))
display(pd.DataFrame(coverage))
for target in targets:
    audit = pd.read_parquet(experiment / target / "test/possession_audit.parquet")
    print(target, "possession eligibility:")
    display(audit.groupby("status").size().rename("possessions").to_frame())

,target,matches,prefixes,possessions,positive_possessions,unweighted_positive_rate,weighted_positive_rate
0,shot,218,290909,51234,11084,0.282267,0.216341
1,goal,218,290909,51234,5421,0.142258,0.105809


shot possession eligibility:


,possessions
status,
eligible,51234
no_pre_shot_action,29
preprocessing_review,3629


goal possession eligibility:


,possessions
status,
eligible,51234
no_pre_shot_action,29
preprocessing_review,3629


## 7. Test probability metrics

**BCE and Brier: lower is better. AP: higher is better.** All metrics here use
possession weights. The constant model predicts development prevalence; its AP
equals weighted test prevalence. AP is a ranking metric, not accuracy.

Model comparisons are made **within each target**. Goal and shot AP values
have different base rates and cannot directly rank target difficulty. Accuracy, precision and
recall use the fixed probability threshold 0.5. No threshold has been tuned.

In [7]:
comparisons = {target: pd.read_parquet(experiment / target / "comparison/metrics.parquet") for target in targets}
metrics = pd.concat([table.assign(target=target) for target, table in comparisons.items()], ignore_index=True)
display(metrics[["target", "model", "log_loss", "brier_score", "average_precision", "accuracy", "precision", "recall", "f1", "balanced_accuracy"]]
        .style.format({column: "{:.6f}" for column in ["log_loss", "brier_score", "average_precision", "accuracy", "precision", "recall", "f1", "balanced_accuracy"]}, na_rep="undefined"))
changes = []
for target, table in comparisons.items():
    indexed = table.set_index("model")
    changes.append(dict(target=target,
        relative_BCE_reduction=1 - indexed.loc["xgboost", "log_loss"] / indexed.loc["logistic", "log_loss"],
        AP_difference=indexed.loc["xgboost", "average_precision"] - indexed.loc["logistic", "average_precision"]))
display(pd.DataFrame(changes).style.format({"relative_BCE_reduction": "{:.2%}", "AP_difference": "{:+.6f}"}))

,target,model,log_loss,brier_score,average_precision,accuracy,precision,recall,f1,balanced_accuracy
0,shot,constant,0.523087,0.169815,0.216341,0.783659,undefined,0.000000,0.000000,0.500000
1,shot,logistic,0.423148,0.134152,0.555929,0.815618,0.672837,0.287535,0.402895,0.624469
2,shot,xgboost,0.398320,0.125400,0.612536,0.829024,0.756987,0.308838,0.438695,0.640734
3,goal,constant,0.337951,0.094666,0.105809,0.894191,undefined,0.000000,0.000000,0.500000
4,goal,logistic,0.297569,0.085786,0.284618,0.894596,0.589756,0.012575,0.024625,0.505770
5,goal,xgboost,0.287580,0.082488,0.342662,0.899497,0.657114,0.104869,0.180872,0.549197


,target,relative_BCE_reduction,AP_difference
0,shot,5.87%,+0.056607
1,goal,3.36%,+0.058044


## 8. Confusion matrices at threshold 0.5

Rows are **actual** classes and columns are **predicted** classes:
`[[true negatives, false positives], [false negatives, true positives]]`.
The raw matrix counts event states. The weighted matrix sums prefix weights;
its fractional entries sum to the number of eligible possessions, but are not
one prediction per possession.

The four matrices report the two XGBoost models with raw and weighted counts. The following table compares all
three models using weighted classification rates. Undefined precision means the
model made no positive predictions. Probability improvement need not imply high
recall at a threshold of 0.5, especially for the rarer goal outcome.

In [8]:
confusions = {target: pd.read_parquet(experiment / target / "comparison/confusion.parquet") for target in targets}
for target, table in confusions.items():
    for weighting in ["unweighted", "possession_weighted"]:
        row = table.loc[table.model.eq("xgboost") & table.weighting.eq(weighting)].iloc[0]
        matrix = pd.DataFrame([[row.tn, row.fp], [row.fn, row.tp]],
            index=[f"Actual no {target}", f"Actual {target}"], columns=[f"Predicted no {target}", f"Predicted {target}"])
        print(target, "—", weighting, "— threshold 0.5")
        display(matrix.style.format("{:.0f}" if weighting == "unweighted" else "{:.2f}"))
rates = pd.concat([table.loc[table.weighting.eq("possession_weighted")].assign(target=target)
                   for target, table in confusions.items()], ignore_index=True)
display(rates[["target", "model", "precision", "recall", "specificity", "f1", "balanced_accuracy"]]
        .style.format({c: "{:.2%}" for c in ["precision", "recall", "specificity", "f1", "balanced_accuracy"]}, na_rep="undefined"))

shot — unweighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,202430,6365
Actual shot,65547,16567


shot — possession_weighted — threshold 0.5


,Predicted no shot,Predicted shot
Actual no shot,39051.08,1098.92
Actual shot,7660.84,3423.16


goal — unweighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,248222,1303
Actual goal,38728,2656


goal — possession_weighted — threshold 0.5


,Predicted no goal,Predicted goal
Actual no goal,45516.36,296.64
Actual goal,4852.51,568.49


,target,model,precision,recall,specificity,f1,balanced_accuracy
0,shot,constant,undefined,0.00%,100.00%,0.00%,50.00%
1,shot,logistic,67.28%,28.75%,96.14%,40.29%,62.45%
2,shot,xgboost,75.70%,30.88%,97.26%,43.87%,64.07%
3,goal,constant,undefined,0.00%,100.00%,0.00%,50.00%
4,goal,logistic,58.98%,1.26%,99.90%,2.46%,50.58%
5,goal,xgboost,65.71%,10.49%,99.35%,18.09%,54.92%


## 9. Calibration

Calibration bands compare mean predicted probability with the observed
target rate using the same possession weights. Approximate agreement is
desirable. This is a diagnostic on the frozen model, not a calibration fit.

Higher bands may contain little weight, so their observed rates can be noisy.
No confidence interval or claim of statistical significance is supplied here.

In [9]:
for target in targets:
    calibration = pd.read_parquet(experiment / target / "comparison/calibration.parquet")
    calibration = calibration.loc[calibration.model.ne("constant")].copy()
    observed = f"observed_{target}_rate"
    calibration["prediction_minus_observed"] = calibration.mean_prediction - calibration[observed]
    print(target)
    display(calibration[["model", "bin_lower", "bin_upper", "examples", "weight_sum", "mean_prediction", observed, "prediction_minus_observed"]]
            .style.format({"weight_sum": "{:.1f}", "mean_prediction": "{:.4f}", observed: "{:.4f}", "prediction_minus_observed": "{:+.4f}"}))

shot


,model,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_shot_rate,prediction_minus_observed
1,logistic,0.000000,0.100000,105430,19872.9,0.0484,0.0606,-0.0122
2,logistic,0.100000,0.200000,70917,11932.3,0.1465,0.1661,-0.0196
3,logistic,0.200000,0.300000,40329,6550.4,0.2465,0.2612,-0.0147
4,logistic,0.300000,0.400000,27121,4691.2,0.3476,0.3325,+0.0151
5,logistic,0.400000,0.500000,19483,3450.5,0.4477,0.4174,+0.0303
6,logistic,0.500000,0.600000,13034,2380.5,0.5460,0.5444,+0.0016
7,logistic,0.600000,0.700000,8115,1391.9,0.6455,0.7227,-0.0772
8,logistic,0.700000,0.800000,4860,815.4,0.7430,0.9126,-0.1696
9,logistic,0.800000,0.900000,1532,143.6,0.8281,0.9491,-0.1211
10,logistic,0.900000,1.000000,88,5.3,0.9191,0.8975,+0.0217


goal


,model,bin_lower,bin_upper,examples,weight_sum,mean_prediction,observed_goal_rate,prediction_minus_observed
1,logistic,0.000000,0.100000,181250,32729.0,0.0461,0.0525,-0.0064
2,logistic,0.100000,0.200000,68629,11556.3,0.1425,0.1485,-0.0060
3,logistic,0.200000,0.300000,26091,4590.6,0.2410,0.2154,+0.0257
4,logistic,0.300000,0.400000,9578,1587.9,0.3434,0.3722,-0.0289
5,logistic,0.400000,0.500000,3972,654.6,0.4398,0.5188,-0.0790
6,logistic,0.500000,0.600000,1160,100.8,0.5325,0.5851,-0.0526
7,logistic,0.600000,0.700000,199,13.3,0.6362,0.6333,+0.0029
8,logistic,0.700000,0.800000,26,1.2,0.7380,0.4550,+0.2830
9,logistic,0.800000,0.900000,4,0.3,0.8558,0.7360,+0.1198
10,xgboost,0.000000,0.100000,174419,33106.5,0.0450,0.0493,-0.0044


## 10. Joint interpretation of shot and goal probabilities

The two classifiers are independently optimised. They do not enforce
`P(goal) <= P(shot)`, even though a goal-positive label is also shot-positive in
these examples. The check below counts any violations in their saved predictions.
It is a diagnostic rather than a reason to change the models using test data.

The results establish a nonlinear current-state benchmark. A future sequence
model can test whether observed action history improves on it. These estimates
are predictive associations; they do not show that changing a particular action
would cause a better outcome, and they are not yet an opposition report.

In [10]:
paired = predictions["shot"][["event_id", "shot_probability", "sample_weight"]].merge(
    predictions["goal"][["event_id", "goal_probability"]], on="event_id", validate="one_to_one")
violations = paired.goal_probability > paired.shot_probability
display(pd.DataFrame({
    "shared_test_prefixes": [len(paired)],
    "goal_probability_exceeds_shot": [int(violations.sum())],
    "weighted_fraction": [paired.loc[violations, "sample_weight"].sum() / paired.sample_weight.sum()],
}))
print("2026 was previously observed for logistic regression. Treat these results as a shared benchmark.")

,shared_test_prefixes,goal_probability_exceeds_shot,weighted_fraction
0,290909,61,0.000453


2026 was previously observed for logistic regression. Treat these results as a shared benchmark.
